<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# Your GeoPandas notebook, scaled with Sedona

Sedona ships a `sedona.spark.geopandas` package that mirrors the public GeoPandas API — same constructors, same method names, same return shapes — but runs on a Spark backend so the same code path scales from a laptop to a cluster. We answer:

> **What does it look like to take a typical GeoPandas script and run it on Sedona?**

Along the way we exercise the methods that landed in 1.8 / 1.9 and that this notebook actually calls — `convex_hull`, `clip_by_rect`, `total_bounds`, `to_geopandas` — and show how to drop into SQL when the GeoPandas-style API doesn't have what you need (`ST_VoronoiPolygons` + `ST_Collect_Agg`, `ST_DistanceSpheroid`).

**Requires Sedona ≥ 1.9.0.** `clip_by_rect` lands in 1.9.0 and the notebook will fail on older versions of the docker image.

Data is the Natural Earth countries shapefile already shipped with the docker image; no network required.

## 1. Connect to Spark through SedonaContext

One difference from the other example notebooks: `sedona.spark.geopandas` runs on top of **pandas-on-Spark** (`pyspark.pandas`), which currently requires Spark's ANSI mode to be off. We set that flag explicitly when building the session.

In [ ]:
from sedona.spark import SedonaContext

config = (
    SedonaContext.builder()
    .master("spark://localhost:7077")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
sedona = SedonaContext.create(config)

## 2. Load a shapefile with the same `read_file` you already know

`sedona.spark.geopandas.read_file` is a drop-in for `geopandas.read_file`. The only twist when pointing at a directory is to declare the format explicitly — the file extension can't be inferred for shapefile bundles.

In [ ]:
from sedona.spark import geopandas as sgpd

countries = sgpd.read_file("data/ne_50m_admin_0_countries_lakes", format="shapefile")
print(f"loaded {len(countries)} countries")
print("columns:", countries.columns.tolist()[:6], "…")
countries[["NAME", "CONTINENT", "POP_EST"]].head(5)

## 3. Filter, then derive — the same idioms as vanilla GeoPandas

Boolean indexing, `.geometry` accessor, `centroid`, `convex_hull`, `area`, and `total_bounds` all work exactly as they do in `geopandas`.

In [ ]:
africa = countries[countries.CONTINENT == "Africa"]
print(f"{len(africa)} African countries")

geom = africa.geometry
print("\nbounding box of the continent:", tuple(round(b, 2) for b in geom.total_bounds))

summary = africa[["NAME"]].copy()
summary["centroid"] = geom.centroid
summary["area_deg2"] = geom.area
summary["hull_area_deg2"] = geom.convex_hull.area
summary.sort_values("area_deg2", ascending=False).head(5)

## 4. Voronoi catchments via `ST_VoronoiPolygons` + `ST_Collect_Agg`

`GeoSeries.voronoi_polygons()` runs a Voronoi tessellation **per row**, which only makes sense if a single row already contains a MultiPoint. To compute one Voronoi diagram from many points, drop into SQL: aggregate every centroid into a single MultiPoint with the `ST_Collect_Agg` aggregator (new in 1.8.1), then call `ST_VoronoiPolygons` on the aggregate.

In [ ]:
africa.spark.frame().createOrReplaceTempView("africa")

voronoi_geom = sedona.sql("""
    SELECT ST_VoronoiPolygons(ST_Collect_Agg(ST_Centroid(geometry))) AS v
    FROM africa
""").first()[0]

stats = sedona.sql("""
    SELECT ST_NumGeometries(v)        AS cells,
           ROUND(ST_Area(v), 2)       AS total_area_deg2,
           ROUND(ST_XMin(v), 2)       AS xmin,
           ROUND(ST_YMin(v), 2)       AS ymin,
           ROUND(ST_XMax(v), 2)       AS xmax,
           ROUND(ST_YMax(v), 2)       AS ymax
    FROM (SELECT ST_VoronoiPolygons(ST_Collect_Agg(ST_Centroid(geometry))) AS v
          FROM africa)
""")
stats.show()

## 5. Clip the Voronoi diagram to a continental bounding rectangle

`clip_by_rect(xmin, ymin, xmax, ymax)` (new in 1.9) is the geopandas-style way to crop. We use it to confine the Voronoi cells to a generous Africa bbox so they line up cleanly with the country polygons in the final plot.

In [ ]:
from shapely.wkt import loads as wkt_loads

voronoi_shapely = (
    wkt_loads(voronoi_geom.wkt) if hasattr(voronoi_geom, "wkt") else voronoi_geom
)
voronoi_cells = sgpd.GeoSeries([g for g in voronoi_shapely.geoms])
print(f"{len(voronoi_cells)} Voronoi cells before clip")

africa_bbox = (-20.0, -36.0, 52.0, 38.0)
clipped = voronoi_cells.clip_by_rect(*africa_bbox)
print(f"{len(clipped)} Voronoi cells after clip_by_rect")
clipped.head(2)

## 6. Hand off to vanilla GeoPandas for plotting

When the data is small enough, `to_geopandas()` materializes a Sedona GeoDataFrame as a vanilla `geopandas.GeoDataFrame` so it can be plotted with the standard `.plot(ax=...)` machinery.

In [ ]:
import matplotlib.pyplot as plt

africa_gp = africa.to_geopandas()
voronoi_gp = clipped.to_geopandas()

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
africa_gp.plot(ax=ax, color="#fdf6e3", edgecolor="#586e75", linewidth=0.6)
voronoi_gp.boundary.plot(ax=ax, color="#dc322f", linewidth=0.4, alpha=0.8)
africa_gp.geometry.centroid.plot(ax=ax, color="#dc322f", markersize=4)
ax.set_title("Africa: Voronoi catchments around country centroids")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_xlim(africa_bbox[0], africa_bbox[2])
ax.set_ylim(africa_bbox[1], africa_bbox[3])
fig.tight_layout()
fig

## 7. Drop into SQL whenever you need a function the API doesn't expose

`<gdf>.spark.frame()` returns the underlying Spark DataFrame, so the entire `ST_*` SQL catalog is one `createOrReplaceTempView` away. Here we ask which African countries' centroids are closest to (0°N, 0°E) using `ST_DistanceSpheroid` (great-circle distance in metres), without leaving the data we already loaded with the geopandas API.

In [ ]:
africa.spark.frame().createOrReplaceTempView("africa")

closest = sedona.sql("""
    SELECT NAME,
           ROUND(ST_DistanceSpheroid(
               ST_Centroid(geometry),
               ST_Point(0.0, 0.0)
           ) / 1000.0, 1) AS km_from_origin
    FROM africa
    ORDER BY km_from_origin ASC
    LIMIT 10
""")
closest.show(truncate=40)

## What just happened?

We took a small GeoPandas-style script and ran it on Sedona's Spark backend without rewriting it as Spark SQL:

1. `read_file(..., format="shapefile")` loaded the Natural Earth countries shapefile into a Sedona GeoDataFrame.
2. Vanilla GeoPandas idioms — boolean filtering, `.geometry`, `.centroid`, `.convex_hull`, `.area`, `.total_bounds` — produced the per-country summary.
3. SQL filled in the gap when the geopandas-on-Spark API didn't have an aggregator: `ST_VoronoiPolygons(ST_Collect_Agg(ST_Centroid(geometry)))` produced a single Voronoi tessellation from many points.
4. `clip_by_rect`, also new in 1.9, cropped the result to a continent.
5. `to_geopandas()` materialized the small final result as a vanilla `geopandas.GeoDataFrame` for plotting.
6. `gdf.spark.frame()` exposed the Spark DataFrame so we could mix in arbitrary SQL — for example `ST_DistanceSpheroid` — without leaving the same dataset.

The same code path runs on a multi-node Spark cluster against billion-row datasets; only the `master(...)` URL changes.